# Snowball Strategy

In [ ]:
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from aws import AthenaDataSource
from core import (
    BacktestTaskDefinition,
    CurrencyPair,
    EventBus,
    InMemoryTaskRepository,
    LogLevel,
    RecordingEventHandler,
    StrategyEvent,
    StrategyParameters,
    TaskManager,
    TaskProfilingConfig,
    TickGranularity,
    TickGranularityFilter,
    TqdmProgressReporter,
    configure_logging,
)
from snowball import SnowballStrategy

configure_logging(level=LogLevel.WARNING)

## Parameters

In [ ]:
INSTRUMENT = CurrencyPair.of("USD_JPY")
START_AT = datetime(2026, 1, 1, tzinfo=UTC)
END_AT = datetime(2026, 1, 31, 23, 59, 59, 999999, tzinfo=UTC)
SAMPLE_GRANULARITY = TickGranularity.TICK
PROFILE_DIR = Path("profiles") / "snowball"

data_source = AthenaDataSource.from_env().with_filters(
    TickGranularityFilter.of(SAMPLE_GRANULARITY),
)

parameters = SnowballStrategy.normalize_parameters(
    StrategyParameters.of(
        sizing={
            "base_units": "1000",
        },
        grid={
            "max_retracements_per_layer": 10,
            "max_layers": 10,
            "refill": {
                "enabled": True,
            },
        },
        cycle={
            "hedging_enabled": True,
            "reseed_when_all_positions_pending_rebuild": True,
        },
        forward={
            "take_profit_pips": "20",
        },
        counter={
            "interval": {
                # choices: "constant", "additive", "subtractive"
                #          "multiplicative", "divisive", "manual"
                "mode": "divisive",
                "head_pips": "30",
                "tail_pips": "10",
                "gamma": "1.3",
            },
            "take_profit": {
                "mode": "weighted_avg"
            },
        },
        stop_loss={
            "enabled": True,
            # choices: "auto", "distance"
            "mode": "auto",
        },
        rebuild={
            "enabled": True,
            "price": {
                # choices: "original_entry_price", "stop_loss_exit_price"
                "entry_price_mode": "stop_loss_exit_price",
            },
            "stop_loss": {
                # choices: "same_price", "same_distance", "manual_distance"
                "mode": "same_distance",
            },
            "take_profit": {
                # choices: "same_price", "same_distance", "progressive_distance"
                "mode": "same_distance"
            },
        },
        protection={
            "shrink_enabled": False,
            "shrink_start_margin_percent": "70",
            "shrink_target_margin_percent": "50",
            "emergency_enabled": False,
            "emergency_margin_percent": "95",
        },
        account={
            "currency": "USD",
            "balance": {
                "amount": "10000",
                "currency": "USD",
            },
            "margin_rate": "0.04",
            "quote_to_account_rate": "1",
        },
    )
)

## Executuin

In [ ]:
definition = BacktestTaskDefinition(
    name=f"{INSTRUMENT} Snowball Athena backtest",
    instrument=INSTRUMENT,
    start_at=START_AT,
    end_at=END_AT,
    parameters=parameters,
)

strategy = SnowballStrategy(
    name="snowball",
    parameters=definition.parameters,
)

recorder = RecordingEventHandler()
event_bus = EventBus(handlers=[recorder])

with TaskManager(
    repository=InMemoryTaskRepository(),
    event_bus=event_bus,
    profiling=TaskProfilingConfig(
        enabled=True,
        cprofile=True,
        cprofile_output_path=PROFILE_DIR,
        pyinstrument=True,
        pyinstrument_output_path=PROFILE_DIR,
    ),
) as manager:
    run = manager.start_backtest(definition, data_source=data_source, strategy=strategy)
    final_task = run.wait(progress=TqdmProgressReporter())

print(f"Task {final_task.id} finished with status: {final_task.status.value}")
print(f"Events recorded: {len(recorder.events)}")

## Results

In [ ]:
events = [event for event in recorder.events if isinstance(event, StrategyEvent)]


def meta(event: StrategyEvent, key: str):
    return event.metadata.get(key)


events_df = pd.DataFrame(
    {
        "timestamp": event.timestamp,
        "display_id": event.display_id,
        "units": float(event.units) if event.units is not None else None,
        "price": str(event.price) if event.price is not None else None,
        "rule": event.reason.rule_id,
        "cycle_id": meta(event, "cycle_id"),
        "direction": meta(event, "direction"),
        "entry_role": meta(event, "entry_role"),
        "layer_number": meta(event, "layer_number"),
        "slot_number": meta(event, "slot_number"),
        "build_number": meta(event, "build_number"),
        "close_reason": meta(event, "close_reason"),
        "is_rebuild": meta(event, "is_rebuild"),
        "planned_entry_price": meta(event, "planned_entry_price"),
        "filled_entry_price": meta(event, "filled_entry_price"),
        "planned_take_profit_price": meta(event, "planned_take_profit_price"),
        "filled_take_profit_price": meta(event, "filled_take_profit_price"),
        "planned_stop_loss_price": meta(event, "planned_stop_loss_price"),
        "filled_stop_loss_price": meta(event, "filled_stop_loss_price"),
        "planned_rebuild_price": meta(event, "planned_rebuild_price"),
        "filled_rebuild_price": meta(event, "filled_rebuild_price"),
        "realized_pl": meta(event, "realized_pl"),
    }
    for event in events
)
events_df

In [ ]:
profile = run.profile()
profile_df = profile.to_dataframe()
profile_counters_df = profile.counters_to_dataframe()
profile_outputs = {
    "cprofile": profile.cprofile_output_path,
    "pyinstrument": profile.pyinstrument_output_path,
    "snakeviz": f"uv run snakeviz {profile.cprofile_output_path}",
    "tuna": f"uv run tuna {profile.cprofile_output_path}",
    "html": f"open {profile.pyinstrument_output_path}",
}
profile_outputs